<a href="https://colab.research.google.com/github/mlbyhoang/customer-segmentation/blob/main/notebooks/02_exploratory_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 – Exploratory Data Analysis

## Objective

Explore customer demographics, contract characteristics,
product categories, and profitability to identify patterns
relevant to customer segmentation.

The original dataset is proprietary and is not included
in this repository.

In [ ]:
# Import Libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
from PIL import Image as PILImage
from IPython.display import display
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import Image

warnings.filterwarnings('ignore')

In [ ]:
# Load prepared dataset
DATA_PATH = Path("../data/customers_prepared.csv")

df = pd.read_csv(DATA_PATH)

In [ ]:
# Dataset dimensions and data types
print(f"Dataset shape: {df.shape}")
df.info()

# Summary statistics
display(df.describe(include="all"))

# Missing values
display(df.isna().sum())


In [ ]:
# Customer Demographics

fig, ax = plt.subplots(figsize=(9, 5))

sns.histplot(
    data=df,
    x="customer_age",
    bins=25,
    kde=True,
    ax=ax
)

ax.set(
    title="Customer Age Distribution",
    xlabel="Customer Age",
    ylabel="Frequency"
)

plt.tight_layout()
plt.show()


In [ ]:
# Contract & Product Characteristics

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(
    data=df,
    x="contract_duration",
    ax=axes[0]
)

axes[0].set_title("Contract Duration")
axes[0].set_xlabel("Duration (Months)")

sns.countplot(
    data=df,
    x="device_category",
    ax=axes[1]
)

axes[1].set_title("Device Categories")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


In [ ]:
# Contract Distribution

fig, ax = plt.subplots(figsize=(9, 5))

sns.histplot(
    data=df,
    x="monthly_fee",
    bins=30,
    ax=ax
)

ax.set(
    title="Monthly Fee Distribution",
    xlabel="Monthly Fee",
    ylabel="Frequency"
)

plt.tight_layout()
plt.show()


In [ ]:
# Profitability Analysis

fig, ax = plt.subplots(figsize=(9, 5))

sns.boxplot(
    data=df,
    x="device_category",
    y="profitability",
    showfliers=False,
    ax=ax
)

ax.set(
    title="Profitability by Device Category",
    xlabel="Device Category",
    ylabel="Profitability"
)

plt.tight_layout()
plt.show()


In [ ]:
# Correlations

# Select numerical features
numerical_columns = [
    "customer_age",
    "contract_duration",
    "monthly_fee",
    "upfront_payment",
    "sim_only",
    "profitability"
]

# Calculate Pearson correlation matrix
corr_matrix = df[numerical_columns].corr()

# Visualize correlations
plt.figure(figsize=(10, 6))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0
)

plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()



In [ ]:
# Date Transformation
df_sh['erstellungsdatum'] = pd.to_datetime(df_sh['erstellungsdatum'])

df_sh['jahr'] = df_sh['erstellungsdatum'].dt.year
df_sh['quartal'] = df_sh['erstellungsdatum'].dt.quarter
df_sh['monat'] = df_sh['erstellungsdatum'].dt.month
df_sh['woche'] = df_sh['erstellungsdatum'].dt.isocalendar().week
df_sh['monatstag'] = df_sh['erstellungsdatum'].dt.day
df_sh['wochentag'] = pd.to_datetime(df['erstellungsdatum']).dt.weekday

In [ ]:
print(df_sh['jahr'].unique())

In [ ]:
# Aggregate annual profitability
annual_profitability = (
    df.groupby("year")["profitability"]
    .agg(["sum", "median"])
    .reset_index()
    .sort_values("year")
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = [
    ("sum", "Total Profitability"),
    ("median", "Median Profitability")
]

for ax, (metric, title) in zip(axes, metrics):
    sns.lineplot(
        data=annual_profitability,
        x="year",
        y=metric,
        marker="o",
        ax=ax
    )

    # Add linear trend only when enough years are available
    if len(annual_profitability) >= 3:
        z = np.polyfit(
            annual_profitability["year"],
            annual_profitability[metric],
            1
        )
        trend = np.poly1d(z)

        ax.plot(
            annual_profitability["year"],
            trend(annual_profitability["year"]),
            linestyle="--",
            label="Linear Trend"
        )
        ax.legend()

    ax.set(
        title=title,
        xlabel="Year",
        ylabel="Profitability"
    )
    ax.set_xticks(annual_profitability["year"])
    ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()
